# Lesson 4d — Full Transformer block

Phase 2 of the Karpathy track: build PRAGMA incrementally. Same data and task throughout L4a-L4d, with one piece added each lesson.

Previous lesson result: **L4c naked attention: 53.6% acc, 0.902 CE (worse! needs residual + FFN + LN)**

Runnable version of `04d_*.py`.


## Same dataset

In [ ]:
import math, random, torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0); random.seed(0)

KEYS   = ["pet", "action", "place"]
VALUES = ["dog", "cat", "fish", "eat", "sleep", "play", "garden", "couch", "bowl"]
PAD, MASK = "<pad>", "<mask>"
vocab  = [PAD, MASK] + KEYS + VALUES
tok2id = {t: i for i, t in enumerate(vocab)}
V = len(vocab)
RULES = {
    "dog":  {"action": ["eat", "play"],   "place": ["garden", "bowl"]},
    "cat":  {"action": ["sleep", "play"], "place": ["couch", "bowl"]},
    "fish": {"action": ["eat", "sleep"],  "place": ["bowl"]},
}

def random_event():
    pet = random.choice(list(RULES))
    act = random.choice(RULES[pet]["action"])
    plc = random.choice(RULES[pet]["place"])
    return {"pet": pet, "action": act, "place": plc}

def encode_event(ev):
    ids = []
    for k in KEYS:
        ids.append(tok2id[k]); ids.append(tok2id[ev[k]])
    return ids

def make_mlm_example(ev, mask_field):
    ids = encode_event(ev)
    pos = 2 * KEYS.index(mask_field) + 1
    tgt = ids[pos]; ids[pos] = tok2id[MASK]
    return ids, pos, tgt

def build_dataset(events):
    Xs, positions, ys = [], [], []
    for ev in events:
        for k in KEYS:
            ids, pos, tgt = make_mlm_example(ev, k)
            Xs.append(ids); positions.append(pos); ys.append(tgt)
    return (torch.tensor(Xs, dtype=torch.long),
            torch.tensor(positions, dtype=torch.long),
            torch.tensor(ys, dtype=torch.long))

train = [random_event() for _ in range(5000)]
test  = [random_event() for _ in range(1000)]
X_tr, pos_tr, y_tr = build_dataset(train)
X_te, pos_te, y_te = build_dataset(test)

## Full block: multi-head attention + FFN + residual + LayerNorm + position embeddings

In [ ]:
D = 16

class MultiHeadAttention(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        assert d % n_heads == 0
        self.d, self.n_heads, self.d_k = d, n_heads, d // n_heads
        self.W_q = nn.Linear(d, d, bias=False)
        self.W_k = nn.Linear(d, d, bias=False)
        self.W_v = nn.Linear(d, d, bias=False)
        self.W_o = nn.Linear(d, d, bias=False)

    def forward(self, x):
        B, L, _ = x.shape
        Q = self.W_q(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, L, self.n_heads, self.d_k).transpose(1, 2)
        attn = F.softmax(Q @ K.transpose(-2, -1) / math.sqrt(self.d_k), dim=-1)
        out = (attn @ V).transpose(1, 2).contiguous().view(B, L, self.d)
        return self.W_o(out)


class TransformerBlock(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.attn  = MultiHeadAttention(d, n_heads)
        self.norm1 = nn.LayerNorm(d)
        self.ff    = nn.Sequential(
            nn.Linear(d, d * 4),
            nn.GELU(),
            nn.Linear(d * 4, d),
        )
        self.norm2 = nn.LayerNorm(d)

    def forward(self, x):
        x = self.norm1(x + self.attn(x))    # Residual + norm around attention.
        x = self.norm2(x + self.ff(x))      # Residual + norm around FFN.
        return x


class FullModel(nn.Module):
    def __init__(self, V, d, n_heads, max_len=16):
        super().__init__()
        self.emb   = nn.Embedding(V, d)
        self.pos   = nn.Embedding(max_len, d)
        self.block = TransformerBlock(d, n_heads)
        self.head  = nn.Linear(d, V)

    def forward(self, ids, positions=None):
        B, L = ids.shape
        pos_ids = torch.arange(L, device=ids.device).expand(B, L)
        h = self.emb(ids) + self.pos(pos_ids)
        h = self.block(h)
        if positions is not None:
            mask_h = h[torch.arange(B), positions]
        else:
            mask_h = h.mean(dim=1)
        return self.head(mask_h)

model = FullModel(V, D, n_heads=2)
print(f"Parameters: {sum(p.numel() for p in model.parameters())}")

## Train

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()

BATCH, N_STEPS = 128, 2000
for step in range(N_STEPS):
    idx = torch.randint(0, X_tr.size(0), (BATCH,))
    logits = model(X_tr[idx], pos_tr[idx])
    loss = loss_fn(logits, y_tr[idx])
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 400 == 0:
        print(f"  step {step:4d}   loss {loss.item():.3f}")

## The Phase-2 leaderboard

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(X_te, pos_te)
    pred = logits.argmax(-1)
    acc = (pred == y_te).float().mean().item()
    ce  = loss_fn(logits, y_te).item()
total = sum(p.numel() for p in model.parameters())

print(f"  L4a (counts):                         63.4% acc,  0.670 CE   (zero params)")
print(f"  L4b (emb + mean-pool + linear):       64.2% acc,  0.558 CE   (~400 params)")
print(f"  L4c (emb + naked attention):          53.6% acc,  0.902 CE   (~1.5k params)")
print(f"  L4d (FULL TRANSFORMER BLOCK):         {acc * 100:.1f}% acc,  {ce:.3f} CE   ({total} params)")

## You just rebuilt `pragma_mini.py` from scratch

What we built is functionally identical to `pragma_mini.py`'s `PragmaMini` class. The PyTorch version uses `nn.TransformerEncoderLayer`; ours uses our own from-scratch block. Same architecture, same recipe.

## Next

[**L5**](lesson_05_pragma_mini.ipynb) — re-read `pragma_mini.py` with full understanding.